In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

In [2]:
"""
Quick Transfer Learning Baseline (ResNet18)
- Trains a simple classifier on cropped single-object images in folder structure:
train_root/
    apple/
    kiwi/
    carrot/
- Saves the model to saved_models/resnet18_baseline.pth
- Keep epochs small for a quick baseline; increase if you have time/GPU.
"""

# ====== Config ======
train_root = "images/train-images-augmented"   # folder with class subfolders
num_epochs = 5                            # keep small for quick baseline
batch_size = 32
lr = 1e-3
num_workers = 2
output_path = "saved_models/resnet18_baseline.pth"
img_size = 100                            # resize for speed

In [3]:
# ====== Setup ======
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# ====== Data ======
transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                        [0.229, 0.224, 0.225]),
])

dataset = datasets.ImageFolder(train_root, transform=transform)
class_names = dataset.classes
num_classes = len(class_names)
print(f"Classes: {class_names}")

loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)

Classes: ['apple', 'carrot', 'kiwi']


In [5]:
# ====== Model ======
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
# Replace final layer
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=lr)

In [6]:
# ====== Train ======
model.train()
for epoch in range(1, num_epochs + 1):
    running_loss = 0.0
    correct = 0
    total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total if total else 0.0
    epoch_acc = 100.0 * correct / total if total else 0.0
    print(f"Epoch {epoch}/{num_epochs} - Loss: {epoch_loss:.4f} - Acc: {epoch_acc:.2f}%")

Epoch 1/5 - Loss: 0.0185 - Acc: 99.31%
Epoch 2/5 - Loss: 0.0000 - Acc: 100.00%
Epoch 3/5 - Loss: 0.0000 - Acc: 100.00%
Epoch 4/5 - Loss: 0.0000 - Acc: 100.00%
Epoch 5/5 - Loss: 0.0000 - Acc: 100.00%


In [7]:
# ====== Save ======
torch.save({
    "model_state_dict": model.state_dict(),
    "class_names": class_names,
    "img_size": img_size,
}, output_path)

print(f"Saved baseline model to {output_path}")

Saved baseline model to saved_models/resnet18_baseline.pth
